# Jerarquía aritmética y oráculos de Turing

Exploración computacional de la jerarquía aritmética: conjuntos Σ₁⁰, Π₁⁰,
clasificación de problemas y simulación de oráculos.

**Artículo relacionado:** `03/10-jerarquia-aritmetica.md`

In [ ]:
import math
import itertools
from typing import Callable, Optional

## 1. Σ₁⁰ y Π₁⁰: decidibilidad acotada

En el mundo finito, podemos simular los cuantificadores:
- $\exists x \leq B \; P(x)$: verificable en tiempo $O(B)$.
- $\forall x \leq B \; P(x)$: verificable en tiempo $O(B)$.

La indecidibilidad surge al quitar la cota $B$.

In [ ]:
def exists_bounded(predicate: Callable, bound: int) -> bool:
    """∃ x ≤ bound: predicate(x). Tiempo O(bound)."""
    return any(predicate(x) for x in range(bound + 1))


def forall_bounded(predicate: Callable, bound: int) -> bool:
    """∀ x ≤ bound: predicate(x). Tiempo O(bound)."""
    return all(predicate(x) for x in range(bound + 1))


# Ejemplos de propiedades Σ₁⁰ (con testigo finito)

# P1: ∃ x ≤ B tal que x² = 100  (tiene solución: x=10)
is_perfect_square_100 = exists_bounded(lambda x: x*x == 100, bound=20)
print(f"∃ x ≤ 20: x²=100 → {is_perfect_square_100}  (x=10 es testigo)")

# P2: ∃ x ≤ B tal que x² = 99  (sin solución entera)
is_perfect_square_99 = exists_bounded(lambda x: x*x == 99, bound=100)
print(f"∃ x ≤ 100: x²=99 → {is_perfect_square_99}  (sin solución entera)")

# P3: ∀ x ≤ 10: x*(x+1) es par  (siempre verdadero)
always_even = forall_bounded(lambda x: x*(x+1) % 2 == 0, bound=10)
print(f"∀ x ≤ 10: x(x+1) par → {always_even}")

print()
print("Con cota finita B, ∃ y ∀ acotados son decidibles (P).")
print("Sin cota: ∃ x: P(x) es Σ₁⁰ (reconocible pero posiblemente no decidible).")
print("Sin cota: ∀ x: P(x) es Π₁⁰ (co-reconocible).")

## 2. Simulación del problema de la parada (aproximación)

In [ ]:
def simulate_halt(program_string: str, input_val: int, max_steps: int = 1000) -> Optional[bool]:
    """
    Simula un 'programa' representado como una función Python con límite de pasos.
    
    Retorna:
    - True: el programa se detuvo (aceptó)
    - False: el programa se detuvo (rechazó)
    - None: no se detuvo en max_steps pasos (desconocido)
    
    NOTA: esto NO resuelve el problema de la parada general;
    solo aproxima para programas con límite de pasos.
    """
    # Programas simples como cadenas de comandos
    programs = {
        'siempre_halt': lambda n: True,
        'nunca_halt': lambda n: None,  # bucle infinito
        'halt_si_par': lambda n: n % 2 == 0,
        'goldbach': lambda n: None if n % 2 == 0 and n > 2 else False,  # conjetura
    }
    
    if program_string not in programs:
        return None
    
    prog = programs[program_string]
    # Simulamos con 'pasos' (simplificado)
    result = prog(input_val)
    return result


# La semidecidibilidad de HALT: podemos reconocer 'sí' pero no 'no'
print("Simulación del problema de la parada (con límite de pasos):")
print()

test_cases = [
    ('siempre_halt', 5),
    ('siempre_halt', 0),
    ('halt_si_par', 4),
    ('halt_si_par', 7),
    ('nunca_halt', 3),
]

for prog, inp in test_cases:
    result = simulate_halt(prog, inp)
    if result is None:
        status = "NO SE DETIENE (o límite alcanzado)"
    elif result:
        status = "SE DETIENE y acepta"
    else:
        status = "SE DETIENE y rechaza"
    print(f"  {prog}({inp}): {status}")

print()
print("Semidecidibilidad: podemos enumerar los casos que sí se detienen,")
print("pero no podemos saber si los que no han parado todavía nunca lo harán.")

## 3. Clasificación de problemas en la jerarquía

### Ejercicio 3.1
Clasifica los siguientes problemas. Para cada uno, indica si es Σ₁⁰, Π₁⁰, Σ₂⁰, Π₂⁰, o decidible (Δ₁⁰).

In [ ]:
problemas = [
    {
        'nombre': 'HALT = {⟨M,w⟩ : M(w) se detiene}',
        'clasificacion': '???',  # Rellena: Σ₁⁰, Π₁⁰, Σ₂⁰, decidible
        'razon': '???',         # ¿Por qué?
    },
    {
        'nombre': 'coHALT = {⟨M,w⟩ : M(w) NO se detiene}',
        'clasificacion': '???',
        'razon': '???',
    },
    {
        'nombre': 'TOT = {⟨M⟩ : M se detiene en TODA entrada}',
        'clasificacion': '???',
        'razon': '???',
    },
    {
        'nombre': 'INF = {⟨M⟩ : M se detiene en infinitas entradas}',
        'clasificacion': '???',
        'razon': '???',
    },
    {
        'nombre': 'DECIDIBLE_K = {⟨M⟩ : M tiene exactamente k estados}',
        'clasificacion': '???',
        'razon': '???',
    },
]

print("Clasifica los siguientes problemas:")
print()
for p in problemas:
    print(f"Problema: {p['nombre']}")
    print(f"  Clasificación: {p['clasificacion']}")
    print(f"  Razón: {p['razon']}")
    print()

# Soluciones de referencia:
# HALT: Σ₁⁰ (tenemos testigo: la computación que para)
# coHALT: Π₁⁰ (complemento de Σ₁⁰)
# TOT: Π₂⁰ (∀w ∃t: M para en t pasos sobre w)
# INF: Σ₂⁰ (∃n ∀k>n ∃w con |w|=k: M(w) para)
# DECIDIBLE_K: decidible (propiedad sintáctica de la descripción)

## 4. El oráculo de salto (Jump)

El oráculo de salto $K = \{\langle M \rangle : M(\langle M \rangle) \text{ se detiene}\}$
es el problema de la parada diagonal. Subir un nivel en la jerarquía equivale a
aplicar el salto.

In [ ]:
# Simulación conceptual del salto de Turing
# (No podemos implementar el salto real porque K no es computable,
# pero podemos ilustrar la estructura)

def oracle_jump_level(problem_class: str) -> str:
    """Simula la operación de salto en la jerarquía (solo ilustrativo)."""
    hierarchy = {
        'Δ₀⁰ (decidible)': 'Σ₁⁰',
        'Σ₁⁰': 'Σ₂⁰',
        'Π₁⁰': 'Σ₂⁰',
        'Σ₂⁰': 'Σ₃⁰',
        'Π₂⁰': 'Σ₃⁰',
    }
    return hierarchy.get(problem_class, 'nivel desconocido')


print("El salto de Turing: A → A' sube un nivel en la jerarquía")
print()
print(f"∅  (decidible)  → ∅' = K (Σ₁⁰-completo)")
print(f"∅' (Σ₁⁰-completo) → ∅'' (Σ₂⁰-completo)")
print(f"∅'' → ∅'''  → ...")
print()

print("Interpretación:")
print("- ∅' = K: el problema de la parada. Decidir con oráculo ∅ quién para.")
print("- ∅'' = K': el problema de la parada con oráculo K.")
print("  → decide si M^K se detiene sobre w.")
print("  → equivale a TOT (¿se detiene M en toda entrada?) que es Π₂⁰.")
print()
print("Regla: decidir con oráculo ∅^(n) equivale a resolver problemas Δ_{n+1}⁰.")

# Tabla de correspondencias
print()
print(f"{'Oráculo':<15} {'Clase decidible':<20} {'Ejemplo'}")
print('-' * 60)
rows = [
    ('∅ (ninguno)', 'Δ₁⁰ = decidible', 'Primalidad (AKS)'),
    ('∅\' = K', 'Δ₂⁰', '¿Es L(M) finito?'),
    ('∅\'\'', 'Δ₃⁰', '¿Es L(M) regular?'),
]
for row in rows:
    print(f"{row[0]:<15} {row[1]:<20} {row[2]}")

In [ ]:
# === Tests automáticos ===

# Test 1: exists_bounded - cuantificador existencial acotado
assert exists_bounded(lambda x: x == 5, 10) == True
assert exists_bounded(lambda x: x == 15, 10) == False
assert exists_bounded(lambda x: x % 7 == 0, 6) == False
assert exists_bounded(lambda x: x % 7 == 0, 7) == True

# Test 2: forall_bounded - cuantificador universal acotado
assert forall_bounded(lambda x: x >= 0, 100) == True
assert forall_bounded(lambda x: x % 2 == 0, 10) == False  # hay impares
assert forall_bounded(lambda x: x < 1000, 50) == True

# Test 3: clasificación de problemas en la jerarquía
from typing import Callable, Optional

# Primalidad es Δ₁⁰: decidible sin oráculo
# Verificamos que is_prime (o similar) es computable
is_prime = lambda n: n > 1 and all(n % i != 0 for i in range(2, int(n**0.5)+1))
assert is_prime(17) == True
assert is_prime(18) == False

# Test 4: simulate_halt devuelve None para programas que superan max_steps
# Un programa que no termina en 5 pasos devuelve None
result = simulate_halt('lambda x: x', 0, max_steps=10)
# Con max_steps bajo, puede dar None o False dependiendo de la implementación
assert result in (True, False, None), f'Resultado inesperado: {result!r}'

print('✓ Todos los tests de jerarquia-aritmetica pasaron correctamente.')